# SAGE: Autonomous Code-Agent Evolution Benchmark
### 100% Free Live Empirical Cloud Runner (Option 1: Local vLLM on Colab T4 GPU)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pratikjain24/SAGE/blob/main/SAGE_Colab_Benchmark.ipynb)

**Key Highlights:**
- 🚀 **Zero API Key Needed**: Downloads `Qwen/Qwen2.5-Coder-3B-Instruct` directly from HuggingFace to Colab's free 15GB Tesla T4 GPU.
- ⚡ **Zero Rate Limits / 429 Errors**: Local vLLM server handles all requests at local GPU speeds.
- 🔬 **100% Genuine Empirical Inference**: Every trajectory episode generates real token-by-token LLM outputs (no synthetic mocks).
- 🛡️ **Full 7-Archetype Benchmark**: Runs G1 (Control), G2 (Prompt Rewriter), G3 (Memory), G4 (Reflection), G5 (Static Verifier), G6 (Regression Guard), G7 (Oracle Verifier).
- 📊 **Honest LaTeX Tables & Figures**: Generates publication-ready `table1_main_results_live.tex`, `table3_statistical_significance_live.tex`, and plots directly from `cycle_metrics.json`.

> **Prerequisite:** Make sure your Colab runtime is set to **GPU (T4)**:  
> *Runtime → Change runtime type → T4 GPU* before running.


## Step 1 — Check GPU Runtime & Clone Repository
> Make sure your runtime is set to **T4 GPU** (*Runtime → Change runtime type → T4 GPU*).

In [ ]:
# Step 1: Verify T4 GPU & Clone SAGE Repository
!nvidia-smi

import os, shutil
from pathlib import Path

# Clean workspace and clone repository
!rm -rf /content/sage
!git clone https://github.com/Pratikjain24/SAGE.git /content/sage
%cd /content/sage

!pip install -q --upgrade pip
!pip install -q -e .
print("✓ Setup complete! Working directory:", os.getcwd())


## Step 2 — Install vLLM & Download Qwen2.5-Coder-3B
> Downloads ~1.8 GB safetensors directly from HuggingFace Hub (no auth required).

In [ ]:
# Step 2: Install vLLM & Download Qwen2.5-Coder-3B
!pip install -q vllm huggingface_hub

# Download 3B model (fits in 15GB T4 VRAM, ~1.8GB safetensors download)
from huggingface_hub import snapshot_download
import os

model_dir = '/content/models/qwen25-coder-3b'
os.makedirs(model_dir, exist_ok=True)
print('Downloading Qwen2.5-Coder-3B-Instruct from HuggingFace...')
snapshot_download(
    repo_id='Qwen/Qwen2.5-Coder-3B-Instruct',
    local_dir=model_dir,
    ignore_patterns=['*.bin', '*.pt'],  # only safetensors
)
print("✓ Model downloaded successfully to", model_dir)


## Step 3 — Launch Local vLLM OpenAI Server in Background
> Starts OpenAI-compatible endpoint at `http://localhost:8000/v1` in ~45–75 seconds.

In [ ]:
# Step 3: Start vLLM Server in Background (Takes ~45-75 seconds on T4)
import subprocess, time, requests, os

print("Launching local vLLM OpenAI-compatible server on port 8000...")
proc = subprocess.Popen([
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "/content/models/qwen25-coder-3b",
    "--served-model-name", "Qwen/Qwen2.5-Coder-3B-Instruct",
    "--port", "8000",
    "--gpu-memory-utilization", "0.90",
    "--max-model-len", "4096",
    "--trust-remote-code",
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait for server to be ready
print("Waiting for vLLM server to start...")
ready = False
for i in range(60):
    try:
        r = requests.get("http://localhost:8000/v1/models", timeout=2)
        if r.status_code == 200:
            models = [m['id'] for m in r.json().get('data', [])]
            print(f"\n✓ vLLM server is READY! Active models: {models}")
            ready = True
            break
    except Exception:
        pass
    time.sleep(3)
    print(f'  ...{(i+1)*3}s elapsed', end='\r')

if not ready:
    raise RuntimeError("vLLM server timed out. Make sure GPU runtime is selected.")

# Point SAGE at local server (no API key needed)
os.environ["OPENAI_API_BASE"] = "http://localhost:8000/v1"
os.environ["OPENAI_API_KEY"] = "EMPTY"
os.environ["GROQ_API_KEY"] = "EMPTY"
print("✓ SAGE configured to connect to local vLLM endpoint: http://localhost:8000/v1")


## Step 4 — Run Full Live Benchmark (All 7 Archetypes, 5 Cycles)
> Runs G1–G7 on local GPU. **Zero API rate limits**, **zero API costs**, **100% genuine LLM token generation**.


In [ ]:
# Step 4: Run Real SAGE Benchmark (All 7 Archetypes, 5 Cycles)
%cd /content/sage

# Runs live empirical benchmark across all 7 archetypes on local T4 GPU.
# Zero API costs, zero rate limits, full authentic token generation.
!python scripts/run_real_experiment.py \
    --config configs/experiments/real_local_strict.yaml \
    --run-id colab_gpu_study \
    --groups G1,G2,G3,G4,G5,G6,G7 \
    --cycles 5 \
    --seeds 42 \
    --train 20 \
    --test 10


## Step 5 — Provenance & Authenticity Audit
> Verifies that every single recorded episode was generated with real LLM tokens.

In [ ]:
# Step 5: Provenance & Authenticity Audit
%cd /content/sage
import json, pathlib

RUN_ID = 'colab_gpu_study'
traj_path = pathlib.Path(f'experiments/runs/{RUN_ID}/trajectory.jsonl')

if not traj_path.exists():
    print(f'Trajectory not found: {traj_path}. Did Step 4 complete?')
else:
    n_end = n_real = n_zero = n_fb = 0
    models = set()
    groups = set()
    total_in = total_out = 0
    for line in open(traj_path, 'r', encoding='utf-8'):
        line = line.strip()
        if not line: continue
        ev = json.loads(line)
        if ev.get('event_type') != 'task_end': continue
        n_end += 1
        p = ev.get('payload', {})
        c = ev.get('cost', {})
        tin = c.get('tokens_in', 0)
        tout = c.get('tokens_out', 0)
        total_in += tin
        total_out += tout
        tok = tin + tout
        if p.get('is_fallback'): n_fb += 1
        if tok == 0: n_zero += 1
        else: n_real += 1
        if p.get('model_name'): models.add(p['model_name'])
        if ev.get('group'): groups.add(ev['group'])
    print('=' * 60)
    print('PROVENANCE & AUTHENTICITY AUDIT')
    print('=' * 60)
    print(f'Total Task Episodes : {n_end}')
    print(f'Real LLM Inferences : {n_real}')
    print(f'Zero-token Calls    : {n_zero}')
    print(f'Mock Fallbacks      : {n_fb}')
    print(f'Total Prompt Tokens : {total_in:,}')
    print(f'Total Gen Tokens    : {total_out:,}')
    print(f'Groups Audited      : {sorted(groups)}')
    print(f'Active Model Names  : {sorted(models)}')
    print('=' * 60)
    if n_fb == 0 and n_zero == 0 and n_real > 0:
        print('✓ PASS: 100% genuine LLM trajectory generated locally on Colab T4 GPU.')
    else:
        print('⚠️ WARNING: Incomplete or contaminated run detected.')


## Step 6 — Generate Honest LaTeX Tables
> Reads directly from `cycle_metrics.json` to generate `table1_main_results_live.tex` and `table3_statistical_significance_live.tex`.


In [ ]:
# Step 6: Generate Honest Publication LaTeX Tables (Bug-Fixed)
%cd /content/sage
import json, pathlib
import numpy as np

RUN_ID = 'colab_gpu_study'
mf = pathlib.Path(f'experiments/runs/{RUN_ID}/results/cycle_metrics.json')
td = pathlib.Path('paper/tables')
td.mkdir(parents=True, exist_ok=True)

if not mf.exists():
    print(f'Not found: {mf}. Run Step 4 first.')
else:
    metrics = json.loads(mf.read_text(encoding='utf-8'))
    META = {
        'G1': 'Frozen Control',
        'G2': 'Prompt Rewriter',
        'G3': 'Memory Accumulator',
        'G4': 'Reflection Agent',
        'G5': 'Static Verifier',
        'G6': 'Regression Guard',
        'G7': 'Oracle Verifier',
    }
    groups = sorted(set(m['group'] for m in metrics))
    seeds = sorted(set(m.get('seed', 42) for m in metrics))
    T_max = max(m.get('cycle', 0) for m in metrics) if metrics else 0
    seed_str = ', '.join(str(s) for s in seeds)

    # Build Table 1 rows directly from live metrics
    rows = []
    for grp in groups:
        gm = [m for m in metrics if m['group'] == grp]
        cycs = sorted(set(m['cycle'] for m in gm))
        c0, cT = cycs[0], cycs[-1]
        p0 = float(np.mean([m['success_rate'] for m in gm if m['cycle'] == c0]))
        pT = float(np.mean([m['success_rate'] for m in gm if m['cycle'] == cT]))
        dr = float(np.mean([m['safety_drift']  for m in gm if m['cycle'] == cT]))
        co = float(sum(m.get('cost_usd', 0.0) for m in gm))
        d = pT - p0
        d_s  = f'+{d:.3f}'  if d  >= 0 else f'{d:.3f}'
        dr_s = f'+{dr:.3f}' if dr > 0  else f'{dr:.3f}'
        rows.append(f'{grp} & {META.get(grp, grp)} & {p0:.3f} & {pT:.3f} & {d_s} & {dr_s} & \\${co:.4f} \\\\')

    header = [
        '\\begin{table*}[t]',
        '\\centering',
        '\\small',
        f'% Auto-generated from {mf} — 100% live empirical data',
        f'\\caption{{\\textbf{{Live Empirical Results}} (Local vLLM, Qwen2.5-Coder-3B on T4 GPU). Seeds: ({seed_str}), T={T_max} cycles.}}',
        '\\label{tab:live}',
        '\\begin{tabular}{llccccc}',
        '\\toprule',
        'Group & Mechanism & $P(0)$ & $P(T)$ & $\\Delta P$ & SafetyDrift & Cost(\\$) \\\\',
        '\\midrule',
    ]
    footer = ['\\bottomrule', '\\end{tabular}', '\\end{table*}']
    out1 = td / 'table1_main_results_live.tex'
    out1.write_text('\n'.join(header + rows + footer), encoding='utf-8')
    print(f'Table 1 written: {out1}\n')
    print(out1.read_text(encoding='utf-8'))

    # Table 3: statistical significance
    try:
        from sage.metrics.significance import StatisticalSignificanceAnalyzer
        analyzer = StatisticalSignificanceAnalyzer(metrics, run_id=RUN_ID)
        report = analyzer.run_analysis()
        out3 = td / 'table3_statistical_significance_live.tex'
        out3.write_text(analyzer._generate_latex_table(report), encoding='utf-8')
        print(f'\nTable 3 written: {out3}\n')
        print(out3.read_text(encoding='utf-8'))
    except Exception as e:
        print(f'Table 3 skipped: {e}')


## Step 7 — Render Publication Figures

In [ ]:
# Step 7: Render Publication Figures (PNG)
%cd /content/sage
import json, pathlib, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import Image, display

RUN_ID = 'colab_gpu_study'
mf = pathlib.Path(f'experiments/runs/{RUN_ID}/results/cycle_metrics.json')

if not mf.exists():
    print(f'Not found: {mf}')
else:
    metrics = json.loads(mf.read_text(encoding='utf-8'))
    figs = pathlib.Path('paper/figures')
    figs.mkdir(parents=True, exist_ok=True)
    COLORS = {
        'G1': '#64748b',
        'G2': '#3b82f6',
        'G3': '#10b981',
        'G4': '#f59e0b',
        'G5': '#8b5cf6',
        'G6': '#ec4899',
        'G7': '#ef4444',
    }
    groups = sorted(set(m['group'] for m in metrics))
    for mkey, ylabel, title, fname in [
        ('safety_drift', 'Safety Drift', 'Safety Drift over Cycles (Local GPU)', 'safety_drift_live.png'),
        ('success_rate', 'Success Rate P(T)', 'Capability over Cycles (Local GPU)', 'capability_live.png'),
    ]:
        fig, ax = plt.subplots(figsize=(7.5, 4.5), dpi=150)
        for grp in groups:
            gm = sorted([m for m in metrics if m['group'] == grp], key=lambda x: x['cycle'])
            ax.plot(
                [m['cycle'] for m in gm],
                [m[mkey] for m in gm],
                marker='o',
                label=grp,
                color=COLORS.get(grp, 'gray'),
                linewidth=2.2,
                markersize=6,
            )
        ax.set_xlabel('Evolutionary Cycle', fontsize=11, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=11, fontweight='bold')
        ax.set_title(f'{title} (Qwen2.5-Coder-3B)', fontsize=12, fontweight='bold')
        ax.legend(frameon=True, fontsize=9)
        ax.grid(True, linestyle='--', alpha=0.5)
        fig.tight_layout()
        p = figs / fname
        fig.savefig(p)
        plt.close(fig)
        print(f'Generated Figure: {p}')
        display(Image(str(p)))


## Step 8 — Download Results ZIP
> Packages the run trajectory, generated LaTeX tables, and plots into `SAGE_Colab_Results.zip`.

In [ ]:
# Step 8: Package & Download Clean Results ZIP
import shutil, pathlib, os
from google.colab import files

RUN_ID = 'colab_gpu_study'
pkg = pathlib.Path('/content/sage_export')
pkg.mkdir(exist_ok=True)

# Copy live run trajectory & metrics
shutil.copytree(f'/content/sage/experiments/runs/{RUN_ID}', str(pkg / 'run'), dirs_exist_ok=True)
# Copy newly generated LaTeX tables and publication figures
shutil.copytree('/content/sage/paper/tables',  str(pkg / 'tables'),  dirs_exist_ok=True)
shutil.copytree('/content/sage/paper/figures', str(pkg / 'figures'), dirs_exist_ok=True)

shutil.make_archive('/content/SAGE_Colab_Results', 'zip', pkg)
mb = os.path.getsize('/content/SAGE_Colab_Results.zip') / 1e6
print(f'Packaged {mb:.1f} MB results — downloading...')
files.download('/content/SAGE_Colab_Results.zip')
